In [1]:
import os
import json
import math
import random
import numpy as np
import scipy.sparse as sp
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from collections import defaultdict

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SEED = 2020
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Device: cuda


In [3]:
TRAIN_FILE = "/kaggle/input/datasets/aryalunawat31/gowalla1/train.txt"
TEST_FILE  = "/kaggle/input/datasets/aryalunawat31/gowalla1/test.txt"

SAVE_ROOT = "/kaggle/working/gowalla_table5_part1"
os.makedirs(SAVE_ROOT, exist_ok=True)

In [4]:
DATASET_NAME = "gowalla1"

EMBED_DIM  = 64
N_LAYERS   = 3
LR         = 1e-3
LAMBDA     = 1e-4
BATCH_SIZE = 1024
N_EPOCHS   = 100
TOPK       = 20
EVAL_EVERY = 5
CKPT_EVERY = 20

# Part 1 norms
NORM_MODES = ["l1_l", "l1_r", "l1"]

DISPLAY_NAMES = {
    "l1_l": "LightGCN-L1-L",
    "l1_r": "LightGCN-L1-R",
    "l1":   "LightGCN-L1",
    "l":    "LightGCN-L",
    "r":    "LightGCN-R",
    "sym":  "LightGCN",
}

In [5]:
def load_data(train_file, test_file):
    train_dict = defaultdict(set)
    test_dict  = defaultdict(set)

    with open(train_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            uid = int(parts[0])
            for iid in parts[1:]:
                train_dict[uid].add(int(iid))

    with open(test_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            uid = int(parts[0])
            for iid in parts[1:]:
                test_dict[uid].add(int(iid))

    all_users = set(train_dict.keys()) | set(test_dict.keys())
    all_items = set()

    for items in train_dict.values():
        all_items |= items
    for items in test_dict.values():
        all_items |= items

    n_users = max(all_users) + 1
    n_items = max(all_items) + 1
    return train_dict, test_dict, n_users, n_items

train_dict, test_dict, n_users, n_items = load_data(TRAIN_FILE, TEST_FILE)

n_train = sum(len(v) for v in train_dict.values())
n_test  = sum(len(v) for v in test_dict.values())

print(f"Users        : {n_users}")
print(f"Items        : {n_items}")
print(f"Train inters : {n_train}")
print(f"Test inters  : {n_test}")
print(f"Density      : {n_train / (n_users * n_items):.6f}")

Users        : 29858
Items        : 40981
Train inters : 810128
Test inters  : 217242
Density      : 0.000662


In [6]:
def build_norm_adj(train_dict, n_users, n_items, norm_mode="sym"):
    user_deg = np.zeros(n_users, dtype=np.float32)
    item_deg = np.zeros(n_items, dtype=np.float32)

    edge_users = []
    edge_items = []

    for u, items in train_dict.items():
        user_deg[u] = len(items)
        for i in items:
            item_deg[i] += 1.0
            edge_users.append(u)
            edge_items.append(i)

    rows, cols, vals = [], [], []

    for u, i in zip(edge_users, edge_items):
        du = user_deg[u]
        di = item_deg[i]

        if du <= 0 or di <= 0:
            continue

        ui = n_users + i

        # Correct full bipartite block normalization:
        # A = [[0, R],
        #      [R^T, 0]]

        if norm_mode == "sym":
            # D^{-1/2} A D^{-1/2}
            w_ui = 1.0 / (math.sqrt(du) * math.sqrt(di))
            w_iu = 1.0 / (math.sqrt(du) * math.sqrt(di))

        elif norm_mode == "l":
            # D^{-1/2} A
            w_ui = 1.0 / math.sqrt(du)   # source=user row
            w_iu = 1.0 / math.sqrt(di)   # source=item row

        elif norm_mode == "r":
            # A D^{-1/2}
            w_ui = 1.0 / math.sqrt(di)   # destination=item col
            w_iu = 1.0 / math.sqrt(du)   # destination=user col

        elif norm_mode == "l1":
            # D^{-1} A D^{-1}
            w_ui = 1.0 / (du * di)
            w_iu = 1.0 / (du * di)

        elif norm_mode == "l1_l":
            # D^{-1} A
            w_ui = 1.0 / du
            w_iu = 1.0 / di

        elif norm_mode == "l1_r":
            # A D^{-1}
            w_ui = 1.0 / di
            w_iu = 1.0 / du

        else:
            raise ValueError(f"Unknown norm_mode: {norm_mode}")

        rows.append(u)
        cols.append(ui)
        vals.append(w_ui)

        rows.append(ui)
        cols.append(u)
        vals.append(w_iu)

    n_total = n_users + n_items
    A = sp.coo_matrix(
        (np.array(vals, dtype=np.float32), (np.array(rows), np.array(cols))),
        shape=(n_total, n_total),
        dtype=np.float32
    )

    A = A.tocoo()
    indices = torch.from_numpy(np.vstack([A.row, A.col])).long()
    values  = torch.from_numpy(A.data).float()

    A_tensor = torch.sparse_coo_tensor(
        indices, values, torch.Size(A.shape)
    ).coalesce().to(DEVICE)

    return A_tensor

In [7]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, n_layers, A_norm):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.A_norm = A_norm

        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self):
        E0 = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        all_layers = [E0]

        E = E0
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.A_norm, E)
            all_layers.append(E)

        E_final = torch.stack(all_layers, dim=1).mean(dim=1)
        return E_final[:self.n_users], E_final[self.n_users:]

    def forward(self, users, pos_items, neg_items):
        u_emb, i_emb = self.propagate()
        u  = u_emb[users]
        pi = i_emb[pos_items]
        ni = i_emb[neg_items]
        pos_scores = (u * pi).sum(dim=-1)
        neg_scores = (u * ni).sum(dim=-1)
        return pos_scores, neg_scores

    @torch.no_grad()
    def get_embeddings(self):
        return self.propagate()

In [8]:
def bpr_loss(model, users, pos_items, neg_items, pos_scores, neg_scores, lam):
    mf_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10).mean()

    u0  = model.user_emb(users)
    pi0 = model.item_emb(pos_items)
    ni0 = model.item_emb(neg_items)

    reg = (u0.pow(2).sum() + pi0.pow(2).sum() + ni0.pow(2).sum()) / (2.0 * len(users))
    return mf_loss + lam * reg

In [9]:
print("Building training pair array...")
train_pairs = np.array(
    [(u, i) for u, items in train_dict.items() for i in items],
    dtype=np.int32
)
print(f"Total train pairs: {len(train_pairs):,}")

def sample_batch(train_pairs, train_dict, n_items, batch_size):
    idx = np.random.randint(0, len(train_pairs), batch_size)
    users = train_pairs[idx, 0]
    pos_items = train_pairs[idx, 1]
    neg_items = np.random.randint(0, n_items, batch_size)

    for j in range(batch_size):
        while neg_items[j] in train_dict[users[j]]:
            neg_items[j] = np.random.randint(0, n_items)

    return (
        torch.LongTensor(users).to(DEVICE),
        torch.LongTensor(pos_items).to(DEVICE),
        torch.LongTensor(neg_items).to(DEVICE),
    )

Building training pair array...
Total train pairs: 810,128


In [10]:
@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_items, k=20, batch=512):
    model.eval()
    u_emb, i_emb = model.get_embeddings()
    i_emb_T = i_emb.T

    total_recall = 0.0
    total_ndcg = 0.0
    test_users = [u for u in test_dict if len(test_dict[u]) > 0]

    for start in range(0, len(test_users), batch):
        batch_users = test_users[start:start + batch]
        u_vecs = u_emb[torch.LongTensor(batch_users).to(DEVICE)]
        scores = torch.mm(u_vecs, i_emb_T).cpu().numpy()

        for idx, user in enumerate(batch_users):
            for ti in train_dict.get(user, []):
                scores[idx][ti] = -np.inf

            top_k = np.argpartition(-scores[idx], k)[:k]
            top_k = top_k[np.argsort(-scores[idx][top_k])]

            gt_set = test_dict[user]
            hits = len(set(top_k) & gt_set)
            total_recall += hits / min(len(gt_set), k)

            dcg = sum(
                1.0 / np.log2(r + 2)
                for r, item in enumerate(top_k)
                if item in gt_set
            )
            idcg = sum(
                1.0 / np.log2(r + 2)
                for r in range(min(len(gt_set), k))
            )
            total_ndcg += dcg / idcg if idcg > 0 else 0.0

    n = len(test_users)
    return total_recall / n, total_ndcg / n

In [11]:
@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_items, k=20, batch=512):
    model.eval()
    u_emb, i_emb = model.get_embeddings()
    i_emb_T = i_emb.T

    total_recall = 0.0
    total_ndcg = 0.0
    test_users = [u for u in test_dict if len(test_dict[u]) > 0]

    for start in range(0, len(test_users), batch):
        batch_users = test_users[start:start + batch]
        u_vecs = u_emb[torch.LongTensor(batch_users).to(DEVICE)]
        scores = torch.mm(u_vecs, i_emb_T).cpu().numpy()

        for idx, user in enumerate(batch_users):
            for ti in train_dict.get(user, []):
                scores[idx][ti] = -np.inf

            top_k = np.argpartition(-scores[idx], k)[:k]
            top_k = top_k[np.argsort(-scores[idx][top_k])]

            gt_set = test_dict[user]
            hits = len(set(top_k) & gt_set)
            total_recall += hits / min(len(gt_set), k)

            dcg = sum(
                1.0 / np.log2(r + 2)
                for r, item in enumerate(top_k)
                if item in gt_set
            )
            idcg = sum(
                1.0 / np.log2(r + 2)
                for r in range(min(len(gt_set), k))
            )
            total_ndcg += dcg / idcg if idcg > 0 else 0.0

    n = len(test_users)
    return total_recall / n, total_ndcg / n

# =========================================================
# PLOTTING
# =========================================================
def save_variant_plots(out_dir, variant_name, train_losses, eval_epochs, recall_history, ndcg_history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{variant_name} | Gowalla | K=3 | dim=64", fontsize=14, fontweight="bold")

    axes[0].plot(range(1, len(train_losses) + 1), train_losses, linewidth=1.5)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BPR Loss")
    axes[0].set_title("Training Loss")
    axes[0].grid(alpha=0.3)

    axes[1].plot(eval_epochs, recall_history, marker="o", markersize=4, linewidth=1.5)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel(f"Recall@{TOPK}")
    axes[1].set_title(f"Recall@{TOPK}")
    axes[1].grid(alpha=0.3)

    axes[2].plot(eval_epochs, ndcg_history, marker="s", markersize=4, linewidth=1.5)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel(f"NDCG@{TOPK}")
    axes[2].set_title(f"NDCG@{TOPK}")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(out_dir, "training_curves.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    return plot_path

In [12]:
def save_comparison_plots(results_by_variant, save_root):
    # loss comparison
    plt.figure(figsize=(10, 6))
    for name, res in results_by_variant.items():
        plt.plot(range(1, len(res["train_losses"]) + 1), res["train_losses"], label=name, linewidth=1.5)
    plt.xlabel("Epoch")
    plt.ylabel("BPR Loss")
    plt.title("Loss Comparison | Gowalla | Part 1")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_root, "comparison_loss.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # recall comparison
    plt.figure(figsize=(10, 6))
    for name, res in results_by_variant.items():
        plt.plot(res["eval_epochs"], res["recall_history"], marker="o", label=name, linewidth=1.5)
    plt.xlabel("Epoch")
    plt.ylabel(f"Recall@{TOPK}")
    plt.title("Recall Comparison | Gowalla | Part 1")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_root, "comparison_recall.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # ndcg comparison
    plt.figure(figsize=(10, 6))
    for name, res in results_by_variant.items():
        plt.plot(res["eval_epochs"], res["ndcg_history"], marker="s", label=name, linewidth=1.5)
    plt.xlabel("Epoch")
    plt.ylabel(f"NDCG@{TOPK}")
    plt.title("NDCG Comparison | Gowalla | Part 1")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_root, "comparison_ndcg.png"), dpi=150, bbox_inches="tight")
    plt.close()

In [13]:
def run_variant(norm_mode):
    variant_name = DISPLAY_NAMES[norm_mode]
    out_dir = os.path.join(SAVE_ROOT, norm_mode)
    os.makedirs(out_dir, exist_ok=True)

    print("\n" + "=" * 90)
    print(f"Training {variant_name}")
    print("=" * 90)

    A_norm = build_norm_adj(train_dict, n_users, n_items, norm_mode=norm_mode)
    print(f"Adj matrix built for {variant_name}: {n_users + n_items} x {n_users + n_items}")

    model = LightGCN(n_users, n_items, EMBED_DIM, N_LAYERS, A_norm).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    steps_per_epoch = max(1, len(train_pairs) // BATCH_SIZE)

    train_losses = []
    recall_history = []
    ndcg_history = []
    eval_epochs = []

    best_recall = -1.0
    best_ndcg = -1.0
    best_epoch = -1

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0

        pbar = tqdm(
            range(steps_per_epoch),
            desc=f"{variant_name} | Epoch {epoch:03d}/{N_EPOCHS}",
            leave=False,
            dynamic_ncols=True
        )

        for _ in pbar:
            users, pos_items, neg_items = sample_batch(train_pairs, train_dict, n_items, BATCH_SIZE)

            pos_scores, neg_scores = model(users, pos_items, neg_items)
            loss = bpr_loss(model, users, pos_items, neg_items, pos_scores, neg_scores, LAMBDA)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = epoch_loss / steps_per_epoch
        train_losses.append(avg_loss)

        if epoch % EVAL_EVERY == 0 or epoch == 1:
            recall, ndcg = evaluate(model, train_dict, test_dict, n_items, k=TOPK)
            recall_history.append(recall)
            ndcg_history.append(ndcg)
            eval_epochs.append(epoch)

            flag = "  <-- best" if recall > best_recall else ""
            print(
                f"Epoch {epoch:03d} | loss {avg_loss:.4f} | "
                f"Recall@{TOPK}: {recall:.4f} | NDCG@{TOPK}: {ndcg:.4f}{flag}"
            )

            if recall > best_recall:
                best_recall = recall
                best_ndcg = ndcg
                best_epoch = epoch

                torch.save(
                    {
                        "epoch": epoch,
                        "norm_mode": norm_mode,
                        "variant_name": variant_name,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "recall": recall,
                        "ndcg": ndcg,
                        "embed_dim": EMBED_DIM,
                        "n_layers": N_LAYERS,
                        "lr": LR,
                        "lambda": LAMBDA,
                        "batch_size": BATCH_SIZE,
                    },
                    os.path.join(out_dir, "best_model.pt")
                )

        if epoch % CKPT_EVERY == 0:
            torch.save(
                {
                    "epoch": epoch,
                    "norm_mode": norm_mode,
                    "variant_name": variant_name,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_losses": train_losses,
                    "recall_history": recall_history,
                    "ndcg_history": ndcg_history,
                    "eval_epochs": eval_epochs,
                },
                os.path.join(out_dir, f"ckpt_epoch{epoch:03d}.pt")
            )
            print(f"  [checkpoint saved @ epoch {epoch}]")

    best_ckpt = torch.load(os.path.join(out_dir, "best_model.pt"), map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])

    final_recall, final_ndcg = evaluate(model, train_dict, test_dict, n_items, k=TOPK)

    plot_path = save_variant_plots(
        out_dir, variant_name,
        train_losses, eval_epochs, recall_history, ndcg_history
    )

    result = {
        "dataset": DATASET_NAME,
        "method": variant_name,
        "norm_mode": norm_mode,
        "n_users": n_users,
        "n_items": n_items,
        "n_train": int(n_train),
        "n_test": int(n_test),
        "embed_dim": EMBED_DIM,
        "n_layers": N_LAYERS,
        "lr": LR,
        "lambda": LAMBDA,
        "batch_size": BATCH_SIZE,
        "epochs_trained": N_EPOCHS,
        "best_epoch": int(best_epoch),
        f"recall@{TOPK}": round(final_recall, 4),
        f"ndcg@{TOPK}": round(final_ndcg, 4),
        "train_losses": train_losses,
        "eval_epochs": eval_epochs,
        "recall_history": recall_history,
        "ndcg_history": ndcg_history,
        "plot_path": plot_path,
    }

    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(result, f, indent=2)

    print("-" * 90)
    print(f"FINAL RESULTS | {variant_name}")
    print(f"Best Epoch   : {best_epoch}")
    print(f"Recall@{TOPK} : {final_recall:.4f}")
    print(f"NDCG@{TOPK}   : {final_ndcg:.4f}")
    print(f"Saved to      : {out_dir}")
    print("-" * 90)

    return result


In [14]:
all_results = []
results_by_variant = {}

for norm_mode in NORM_MODES:
    result = run_variant(norm_mode)
    all_results.append({
        "method": result["method"],
        "recall@20": result["recall@20"],
        "ndcg@20": result["ndcg@20"],
        "best_epoch": result["best_epoch"],
    })
    results_by_variant[result["method"]] = result

# Save summary CSV
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(os.path.join(SAVE_ROOT, "summary.csv"), index=False)

# Save pretty table
print("\n" + "=" * 90)
print("PART 1 SUMMARY TABLE")
print("=" * 90)
print(summary_df.to_string(index=False))
print("=" * 90)

# Save comparison plots
save_comparison_plots(results_by_variant, SAVE_ROOT)

# Save global metadata
meta = {
    "dataset": DATASET_NAME,
    "part": "part1",
    "norm_modes": NORM_MODES,
    "display_names": [DISPLAY_NAMES[x] for x in NORM_MODES],
    "embed_dim": EMBED_DIM,
    "n_layers": N_LAYERS,
    "epochs": N_EPOCHS,
    "lr": LR,
    "lambda": LAMBDA,
    "batch_size": BATCH_SIZE,
    "topk": TOPK,
    "save_root": SAVE_ROOT,
}
with open(os.path.join(SAVE_ROOT, "run_config.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nAll outputs saved in: {SAVE_ROOT}")


Training LightGCN-L1-L
Adj matrix built for LightGCN-L1-L: 70839 x 70839


Epoch 001 | loss 0.3475 | Recall@20: 0.0754 | NDCG@20: 0.0591  <-- best


Epoch 005 | loss 0.0899 | Recall@20: 0.1089 | NDCG@20: 0.0921  <-- best


Epoch 010 | loss 0.0664 | Recall@20: 0.1201 | NDCG@20: 0.1010  <-- best


Epoch 015 | loss 0.0553 | Recall@20: 0.1261 | NDCG@20: 0.1070  <-- best


...

Epoch 080 | loss 0.0550 | Recall@20: 0.1353 | NDCG@20: 0.1131  <-- best
  [checkpoint saved @ epoch 80]


Epoch 085 | loss 0.0539 | Recall@20: 0.1355 | NDCG@20: 0.1120  <-- best


Epoch 090 | loss 0.0533 | Recall@20: 0.1369 | NDCG@20: 0.1137  <-- best


Epoch 095 | loss 0.0521 | Recall@20: 0.1380 | NDCG@20: 0.1143  <-- best


Epoch 100 | loss 0.0508 | Recall@20: 0.1379 | NDCG@20: 0.1138
  [checkpoint saved @ epoch 100]
------------------------------------------------------------------------------------------
FINAL RESULTS | LightGCN-L1
Best Epoch   : 95
Recall@20 : 0.1380
NDCG@20   : 0.1143
Saved to      : /kaggle/working/gowalla_table5_part1/l1
------------------------------------------------------------------------------------------

PART 1 SUMMARY TABLE
       method  recall@20  ndcg@20  best_epoch
LightGCN-L1-L     0.1574   0.1297         100
LightGCN-L1-R     0.1505   0.1267         100
  LightGCN-L1     0.1380   0.1143          95

All outputs saved in: /kaggle/working/gowalla_table5_part1
